# B2-020-language-transformers — Practice p07 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** embedding-model-training

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

Optimize the table and the distinct head together, then recompute the loss from the updated objects.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

ATOL = RTOL = 1e-6

def update_embedding_table(table, head, context_ids, target_ids):
    optimizer = torch.optim.SGD([table, *head.parameters()], lr=0.1)
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(head(table[context_ids]), target_ids, reduction="mean")
    loss.backward()
    optimizer.step()
    fresh_loss = F.cross_entropy(head(table[context_ids]), target_ids, reduction="mean")
    return fresh_loss, table

torch.manual_seed(20260812)
table = nn.Parameter(torch.randn(12, 8, dtype=torch.float32))
head = nn.Linear(8, 12, dtype=torch.float32)
assert head.weight.data_ptr() != table.data_ptr()
context_ids = torch.tensor([4, 5, 4, 7, 5, 4], dtype=torch.int64)
target_ids = torch.tensor([5, 4, 7, 4, 4, 5], dtype=torch.int64)
before = table.detach().clone()
initial_loss = F.cross_entropy(head(table[context_ids]), target_ids, reduction="mean").detach()
fresh_loss, updated_table = update_embedding_table(table, head, context_ids, target_ids)

### Answer check

In [ ]:
assert fresh_loss.item() < initial_loss.item()
assert updated_table is table
for row in (4, 5, 7):
    assert not torch.allclose(updated_table[row], before[row], atol=ATOL, rtol=RTOL)
assert torch.allclose(updated_table[11], before[11], atol=ATOL, rtol=RTOL)